[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/05_alumno_supervisado.ipynb)

# MLY1101 · Machine Learning — EA2
## Aprendizaje supervisado: predecir qué detecciones van a fallar

**Resultado de aprendizaje (RA2):** implementa modelos de aprendizaje supervisado para
resolver problemas de predicción, evaluando su desempeño con métricas pertinentes al
contexto del negocio.

---

### De dónde venimos

En la EA1 respondimos *"¿podemos confiar en estos datos?"*. Ahora que están limpios, la
pregunta cambia:

```
Problema → Datos → Exploración → Preprocesamiento → MODELAMIENTO → Evaluación → Interpretación
                                                    └──── aquí estamos hoy ────┘
```

### La pregunta de hoy

> **¿Se puede anticipar qué detecciones van a ser difíciles**, a partir de la geometría del
> objeto y de dónde está?

Si se pudiera, el equipo de percepción sabría **de antemano** en qué situaciones no conviene
confiar en el sensor. Eso vale más que un modelo que clasifica objetos: es información
accionable sobre la propia incertidumbre del sistema.

---

### Por qué no clasificamos el tipo de objeto

Parecía lo natural, y es lo que casi todos proponen. Lo probamos: **se resuelve al 99,98 %**
con cualquier configuración. El generador sortea las dimensiones por tipo de objeto, así que
basta el largo de la caja para acertar.

Un ejercicio donde todo sale perfecto no enseña nada sobre evaluación. Y aprender a evaluar
es exactamente lo que se evalúa hoy.

---

### La idea central de hoy

> **Un modelo no se reporta con una cifra.**

Vas a entrenar un modelo que alcanza casi un 90 % de exactitud. También vas a descubrir que
un modelo que responde **siempre lo mismo, sin mirar los datos**, alcanza un 88,96 %.

Esos dos números juntos son la sesión completa.

---

### Al final de la sesión debes entregar

Un **informe de modelamiento** (última celda) con:

- la justificación de cómo partiste el dataset, con la cifra que lo respalda;
- la comparación de tu modelo contra el baseline, en **dos** métricas;
- el análisis por clase, señalando explícitamente a quién falla el modelo;
- una decisión argumentada: ¿pondrías este modelo en producción?

---
## Preparación del entorno

Ejecuta esta celda primero. Funciona en Google Colab y en Jupyter local.

> Fíjate en algo: no vamos a reescribir la limpieza de la EA1. Vamos a **importar las mismas
> funciones** que ya usamos entonces, que además son las que corren en el pipeline del
> repositorio. Una sola verdad sobre los datos.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"

print("Colab:", EN_COLAB)
print("¿Existe el dataset?:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

import eda  # utilidades de diagnóstico de la EA1

# Los nodos del pipeline del repositorio. Solo necesitan pandas: no hace falta
# instalar Kedro para usarlos.
from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as modelo

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
print("Parámetros disponibles:", sorted(PARAMETROS))

---
# Bloque 1 · Del dato limpio al problema supervisado

Un problema supervisado necesita tres cosas, y ninguna es el algoritmo:

| Pieza | Qué es | Aquí |
|---|---|---|
| **`X`** | Las variables predictoras | Geometría y posición del objeto |
| **`y`** | La etiqueta que se quiere predecir | `detection_difficulty` |
| **El grupo** | La unidad que no se puede partir | `segment_id` |

Esa tercera pieza no aparece en los tutoriales y es la que decide si el resultado es honesto.
La veremos en el bloque 2.

In [ ]:
crudo = pd.read_csv(RUTA_DATOS)

# La limpieza de la EA1, aplicada paso a paso con los mismos nodos del pipeline.
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

print(f"Crudo  : {crudo.shape[0]:,} filas × {crudo.shape[1]} columnas")
print(f"Limpio : {limpio.shape[0]:,} filas × {limpio.shape[1]} columnas")
limpio.head(3)

### ✏️ TODO 1 — ¿Está desbalanceado?

Antes de elegir cualquier métrica hay que saber cómo se reparte la etiqueta. Calcula la
frecuencia absoluta y relativa de `detection_difficulty`.

*Pista: `eda.resumen_desbalance()` de la EA1 hace exactamente esto.*

In [ ]:
# TODO 1: ¿cómo se reparte la etiqueta que queremos predecir?
distribucion = eda.____(limpio["____"])
distribucion

In [ ]:
# Autochequeo
minoritaria = distribucion["pct"].idxmin()
pct_minoritaria = distribucion.loc[minoritaria, "pct"]
assert pct_minoritaria < 20, "revisa: ¿estás mirando la columna detection_difficulty?"
print(f"✅ Clase minoritaria: {minoritaria} con {pct_minoritaria:.1f} % de las filas.")
print("   Recuerda esa cifra: vuelve en el bloque 3 y decide toda la sesión.")

### ✏️ TODO 2 — Qué **no** puede ser una variable

Tres columnas de este dataset no pueden entrar en `X`, cada una por un motivo distinto.
Completa el diccionario diciendo por qué.

*Pista: piensa en qué pasaría si el modelo las usara. Una es un identificador, otra define los
grupos de la partición, y la tercera es de donde el sensor **deriva** la etiqueta.*

In [ ]:
# TODO 2: ¿por qué cada una de estas columnas NO puede ser una variable predictora?
por_que_no_son_variables = {
    "id_interno": "____",
    "segment_id": "____",
    "num_lidar_points": "____",
}
for columna, motivo in por_que_no_son_variables.items():
    print(f"{columna:20s} -> {motivo}")

### ✏️ TODO 3 — Armar la tabla de modelamiento

Usa `modelo.preparar_variables()` con los parámetros del pipeline. Comprueba después qué
columnas quedaron.

In [ ]:
# TODO 3: arma la tabla de modelamiento con los parámetros del pipeline.
CONFIG = PARAMETROS["____"]
FUGA = PARAMETROS["fuga"]

tabla = modelo.____(limpio, CONFIG, FUGA)

print("Variables predictoras:", CONFIG["variables"])
print("Etiqueta             :", CONFIG["objetivo"])
print("Grupo                :", CONFIG["grupo"])
print(f"\nTabla de modelamiento: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
tabla.head(3)

In [ ]:
# Autochequeo
assert CONFIG["objetivo"] in tabla.columns, "falta la etiqueta"
assert CONFIG["grupo"] in tabla.columns, "falta la llave de agrupación"
assert "num_lidar_points" not in CONFIG["variables"], (
    "revisa: num_lidar_points NO debe estar entre las variables predictoras"
)
assert "num_lidar_points" in tabla.columns, (
    "pero sí debe viajar en la tabla: la necesitamos en el bloque 6 para medir la fuga"
)
print("✅ Tabla lista. Fíjate en la sutileza: num_lidar_points está en la TABLA")
print("   pero no entre las VARIABLES. Volvemos a eso en el bloque 6.")

---
# Bloque 2 · ⭐ La partición: dónde se gana o se pierde la honestidad

Todo el mundo sabe que hay que separar entrenamiento y prueba. Casi nadie se pregunta **cómo**.

La respuesta por defecto —`train_test_split` al azar— es correcta solo si las filas son
**independientes entre sí**. Aquí no lo son:

> Una fila es **una detección**. Un segmento son **~20 segundos de grabación** con decenas de
> detecciones que comparten clima, momento del día, ubicación y estado del sensor.

Si partes al azar, detecciones del mismo segmento caen en entrenamiento **y** en prueba. El
modelo puede reconocer el segmento en vez del objeto, y la métrica de prueba mide memoria, no
capacidad de generalizar.

Es **fuga de información por agrupación**, y no da ningún síntoma: el modelo no falla, saca
mejor nota.

### ✏️ TODO 4 — Partir por grupo

Usa `modelo.particionar()`, que agrupa por `segment_id`. Después comprueba cuántos segmentos
quedan a caballo entre las dos partes.

In [ ]:
# TODO 4: parte el dataset agrupando por segmento.
marcada = modelo.____(tabla, CONFIG)

entrena = marcada[marcada["particion"] == "entrenamiento"]
prueba = marcada[marcada["particion"] == "____"]

segmentos_entrena = set(entrena[CONFIG["grupo"]])
segmentos_prueba = set(prueba[CONFIG["grupo"]])

print(f"Entrenamiento: {len(entrena):,} filas ({len(entrena)/len(marcada):.1%})  "
      f"en {len(segmentos_entrena)} segmentos")
print(f"Prueba       : {len(prueba):,} filas ({len(prueba)/len(marcada):.1%})  "
      f"en {len(segmentos_prueba)} segmentos")
print(f"\nSegmentos en AMBAS partes: {len(segmentos_entrena & segmentos_prueba)}")

### ✏️ TODO 5 — La partición ingenua, para comparar

Ahora la vía por defecto: al azar por fila. Cuenta los segmentos compartidos.

In [ ]:
# TODO 5: la partición al azar por fila, para poder comparar.
al_azar = modelo.____(tabla, CONFIG)

az_entrena = set(al_azar.loc[al_azar["particion"] == "entrenamiento", CONFIG["grupo"]])
az_prueba = set(al_azar.loc[al_azar["particion"] == "prueba", CONFIG["grupo"]])

print(f"Partición por grupo   -> segmentos compartidos: {len(segmentos_entrena & segmentos_prueba)}")
print(f"Partición al azar     -> segmentos compartidos: {len(az_entrena & az_prueba)}")

In [ ]:
# Autochequeo
assert len(segmentos_entrena & segmentos_prueba) == 0, (
    "revisa: al partir por grupo NINGÚN segmento puede estar en las dos partes"
)
assert len(az_entrena & az_prueba) > 0, (
    "revisa: la partición al azar debería dejar segmentos compartidos"
)
print(f"✅ Por grupo: 0 segmentos compartidos.  Al azar: {len(az_entrena & az_prueba)}.")

### ✏️ TODO 6 — ¿Y cuánto daño hace?

Aquí viene lo interesante, y no es lo que esperas. Mide el efecto en vez de suponerlo.

In [ ]:
# TODO 6: entrena con las dos particiones y compara.
comparacion = modelo.____(marcada, al_azar, CONFIG)
comparacion

**✍️ Tu respuesta al TODO 6:**

*(doble clic aquí y escribe)*

La diferencia entre las dos particiones es prácticamente nula. Entonces, ¿da igual cómo se
parta? Justifica tu respuesta.

---
# Bloque 3 · ⭐⭐ El baseline: contra qué compites

Antes de entrenar nada, una pregunta incómoda:

> **¿Qué exactitud saca un modelo que no mira los datos?**

Un *baseline* es un modelo deliberadamente estúpido. `DummyClassifier` de scikit-learn tiene
varias estrategias; la más brutal es `most_frequent`: **responde siempre la clase mayoritaria**,
pase lo que pase.

No es un ejercicio retórico. Es la referencia sin la cual **ninguna métrica significa nada**.

### ✏️ TODO 7 — Antes de ejecutar, apuesta

Escribe tu predicción **antes** de correr la celda siguiente:

**Creo que el baseline `most_frequent` sacará una exactitud de:** `____ %`

*(No hagas trampa. Escríbelo primero.)*

In [ ]:
# TODO 7: entrena el baseline que responde siempre la clase mayoritaria.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

X_entrena = entrena[CONFIG["variables"]]
y_entrena = entrena[CONFIG["objetivo"]]
X_prueba = prueba[CONFIG["variables"]]
y_prueba = prueba[CONFIG["objetivo"]]

tonto = DummyClassifier(strategy="____", random_state=42)
tonto.fit(X_entrena, y_entrena)

reporte_tonto = classification_report(
    y_prueba, tonto.predict(X_prueba), output_dict=True, zero_division=0
)

print(f"Baseline 'responder siempre lo mismo':")
print(f"   exactitud : {reporte_tonto['accuracy']:.4f}")
print(f"   F1-macro  : {reporte_tonto['macro avg']['f1-score']:.4f}")

### ✏️ TODO 8 — ¿De dónde sale ese número?

El baseline saca casi un 89 % de exactitud **sin mirar una sola variable**. Comprueba que no es
casualidad: compara esa exactitud con la proporción de la clase mayoritaria en el conjunto de
prueba.

In [ ]:
# TODO 8: ¿por qué el baseline saca justo ese número?
pct_mayoritaria = (y_prueba == y_prueba.value_counts().____()).mean()

print(f"Exactitud del baseline          : {reporte_tonto['accuracy']:.4f}")
print(f"Proporción de la clase mayoritaria: {pct_mayoritaria:.4f}")
print(f"\n¿Son el mismo número?: {np.isclose(reporte_tonto['accuracy'], pct_mayoritaria)}")

In [ ]:
# Autochequeo
assert np.isclose(reporte_tonto["accuracy"], pct_mayoritaria), (
    "revisa: la exactitud del baseline ES la proporción de la clase mayoritaria"
)
print("✅ La exactitud del baseline no es un resultado: es la proporción de la clase")
print("   mayoritaria, con otro nombre. En un problema desbalanceado, la exactitud")
print("   te dice cuán desbalanceado está, no cuán bueno es tu modelo.")

---
# Bloque 4 · Entrenar un modelo de verdad

Ahora sí. Un **bosque aleatorio**: muchos árboles de decisión entrenados sobre muestras
distintas, que votan.

Fíjate en algo mientras lo escribes: **entrenar son tres líneas**. Todo lo difícil de esta
sesión está alrededor —cómo partiste, contra qué comparas, cómo mides—, no aquí.

### Un detalle que no es decorativo: `class_weight="balanced"`

`LEVEL_2` es el 11 % de los datos. Sin compensar, el modelo aprende que **ignorar la clase
minoritaria casi no penaliza** el acierto global. Y en un sistema de percepción de un vehículo,
la clase minoritaria es justo la que importa.

### ✏️ TODO 9 — Entrenar

Usa `modelo.entrenar()`, que aplica los parámetros del pipeline.

In [ ]:
# TODO 9: entrena el clasificador con la partición POR GRUPO.
clasificador = modelo.____(marcada, CONFIG)

print("Modelo:", type(clasificador).__name__)
print("Árboles:", clasificador.n_estimators, "| profundidad máxima:", clasificador.max_depth)
print("Variables que vio:", list(clasificador.feature_names_in_))

### ✏️ TODO 10 — El momento de la verdad

Compara tu modelo con el baseline. En **exactitud** y en **F1-macro**.

In [ ]:
# TODO 10: compara tu modelo con el baseline en DOS métricas.
reporte = classification_report(
    y_prueba, clasificador.____(X_prueba), output_dict=True, zero_division=0
)

resultados = pd.DataFrame(
    [
        {"modelo": "baseline (siempre lo mismo)",
         "exactitud": round(reporte_tonto["accuracy"], 4),
         "f1_macro": round(reporte_tonto["macro avg"]["____"], 4)},
        {"modelo": "bosque aleatorio",
         "exactitud": round(reporte["accuracy"], 4),
         "f1_macro": round(reporte["macro avg"]["____"], 4)},
    ]
)
resultados["mejora"] = (resultados["exactitud"] - resultados.loc[0, "exactitud"]).round(4)
resultados

In [ ]:
# Autochequeo
ganancia_exactitud = reporte["accuracy"] - reporte_tonto["accuracy"]
ganancia_f1 = reporte["macro avg"]["f1-score"] - reporte_tonto["macro avg"]["f1-score"]

assert ganancia_exactitud < 0.02, (
    "revisa: la ganancia en exactitud debería ser pequeñísima. ¿Partiste por grupo?"
)
assert ganancia_f1 > 0.15, "revisa: en F1-macro la ganancia sí debería ser grande"
print(f"✅ Ganancia en exactitud: {ganancia_exactitud:+.4f}  ({ganancia_exactitud*100:+.2f} puntos)")
print(f"   Ganancia en F1-macro : {ganancia_f1:+.4f}")
print()
print("   Dos métricas sobre el MISMO modelo, con conclusiones opuestas.")
print("   Una dice que no sirvió de nada. La otra, que mejoró muchísimo.")
print("   El bloque 5 explica cuál tiene razón.")

**✍️ Tu respuesta al TODO 10:**

*(doble clic aquí y escribe)*

Tu modelo mejora la exactitud del baseline en menos de un punto porcentual. ¿Fue un fracaso?
¿Qué mirarías para decidirlo?

---
# Bloque 5 · ⭐⭐ Evaluar por clase: ¿a quién le falla?

Un promedio global oculta a la minoría. Para saber qué hace de verdad un modelo hay que mirar
clase por clase:

| Métrica | Qué responde | Fórmula |
|---|---|---|
| **Precisión** | De lo que predije como `LEVEL_2`, ¿cuánto lo era? | VP / (VP + FP) |
| **Recall** | De todo lo que era `LEVEL_2`, ¿cuánto encontré? | VP / (VP + FN) |
| **F1** | La media armónica de las dos | 2·P·R / (P + R) |

**Cuál importa depende del negocio, no de la estadística.** Aquí queremos anticipar detecciones
difíciles para no confiarnos: si se nos escapan, el sistema confía en una detección mala. Eso
apunta al **recall**.

### ✏️ TODO 11 — El reporte por clase

In [ ]:
# TODO 11: métricas por clase, no solo el promedio.
metricas = modelo.____(clasificador, marcada, CONFIG)
metricas

### ✏️ TODO 12 — La matriz de confusión

Las filas son lo que **era**; las columnas, lo que el modelo **dijo**. La diagonal son los
aciertos.

In [ ]:
# TODO 12: la matriz de confusión.
confusion = modelo.____(clasificador, marcada, CONFIG)
print(confusion, "\n")

fig, ejes = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ejes)
ejes.set_xlabel("Predicho")
ejes.set_ylabel("Real")
ejes.set_title("Matriz de confusión")
plt.tight_layout()
plt.show()

### ✏️ TODO 13 — Poner número al daño

De la matriz, calcula cuántas detecciones difíciles se le escaparon al modelo y qué porcentaje
del total de difíciles representan.

In [ ]:
# TODO 13: ¿cuántas detecciones difíciles se le escaparon?
difciles_perdidas = confusion.loc["____", "____"]
difciles_totales = confusion.loc["LEVEL_2"].sum()
falsas_alarmas = confusion.loc["____", "____"]

print(f"Detecciones difíciles en el conjunto de prueba : {difciles_totales:,}")
print(f"Que el modelo NO detectó como difíciles        : {difciles_perdidas:,}"
      f"  ({100*difciles_perdidas/difciles_totales:.1f} %)")
print(f"Falsas alarmas (fáciles marcadas como difíciles): {falsas_alarmas:,}")

In [ ]:
# Autochequeo
recall_l2 = metricas.set_index("clase").loc["LEVEL_2", "recall"]
assert np.isclose(1 - difciles_perdidas / difciles_totales, recall_l2, atol=1e-3), (
    "revisa: lo que NO se escapó, sobre el total, ES el recall de LEVEL_2"
)
assert recall_l2 < 0.5, "revisa: el recall de la clase minoritaria debería ser bajo"
print(f"✅ Recall de LEVEL_2: {recall_l2:.4f}")
print(f"   El modelo encuentra {recall_l2:.0%} de las detecciones difíciles.")
print(f"   Se le escapan {100*difciles_perdidas/difciles_totales:.0f} de cada 100.")

### ✏️ TODO 14 — La decisión

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. Con estas cifras, ¿pondrías este modelo a decidir si el vehículo confía o no en una
   detección? Justifica.
2. ¿Qué preferirías: subir el recall a costa de más falsas alarmas, o al revés? ¿Por qué, en
   **este** dominio?

---
# Bloque 6 · ⭐ La fuga que sí se nota

En el bloque 2 vimos una fuga que existía pero no movía la métrica. Esta es la contraria: se
mide y duele.

`detection_difficulty` **la asigna el sensor a partir de `num_lidar_points`**. Meter esa columna
entre las predictoras no es informativo: es contarle al modelo la respuesta con otras palabras.

Y aquí está lo traicionero: **el modelo mejora**. No hay ningún síntoma, solo un resultado
sospechosamente bueno.

### ✏️ TODO 15 — Medir la fuga

In [ ]:
# TODO 15: ¿cuánto infla la métrica incluir la variable de la que sale la etiqueta?
efecto_fuga = modelo.____(marcada, CONFIG, FUGA)
efecto_fuga

In [ ]:
# Autochequeo
inflacion = efecto_fuga.loc[1, "inflacion_f1_macro"]
assert inflacion > 0, "revisa: incluir la variable derivada debería SUBIR la métrica"
print(f"✅ Incluir num_lidar_points sube el F1-macro en {inflacion:+.4f}.")
print("   El modelo parece mejor y no sirve para nada: en el momento en que")
print("   quisieras predecir la dificultad, ya tendrías la dificultad.")

### ✏️ TODO 16 — La pregunta que detecta la fuga

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

Existe **una sola pregunta** que hay que hacerle a cada variable predictora para detectar este
tipo de fuga. No es estadística. Escríbela con tus palabras.

*Pista: piensa en el momento en que el modelo se usaría de verdad.*

---
# Bloque 7 · ¿Qué aprendió el modelo?

Un modelo que acierta y no se puede explicar sirve para poco: nadie firma una decisión que no
entiende. El bosque aleatorio permite ver **qué variables usó más**.

Cuidado con la interpretación: la importancia dice **cuánto usó** el modelo cada variable, no
que esa variable *cause* nada.

### ✏️ TODO 17 — Importancia de variables

In [ ]:
# TODO 17: ¿qué variables usó más el modelo?
importancia = (
    pd.Series(clasificador.____, index=CONFIG["variables"])
    .sort_values(ascending=False)
    .round(4)
)
print(importancia.to_string(), "\n")

importancia.sort_values().plot.barh(figsize=(6, 3.5), title="¿Qué usó el modelo?")
plt.tight_layout()
plt.show()

**✍️ Tu respuesta al TODO 17:**

*(doble clic aquí y escribe)*

Mira las dos variables más importantes. ¿Tiene sentido de dominio que sean esas? Explica el
mecanismo físico que lo justificaría.

---
# Cierre · Informe de modelamiento

Esta es la entrega de la EA2. Máximo una página.

---

### El problema

**Qué queríamos predecir:** `____`
**Por qué le importa a alguien:** `____`
**Tipo de problema:** `____` *(clasificación binaria / multiclase / regresión)*

### Cómo partí los datos

**Estrategia:** `____`
**Segmentos compartidos entre entrenamiento y prueba:** `____`
**Por qué no partí al azar:** `____`

> Ojo con esta última: la diferencia medida fue casi nula. Si tu argumento es "porque da mejor
> resultado", vuelve a leer el bloque 2.

### Contra qué comparé

| Modelo | Exactitud | F1-macro |
|---|---|---|
| Baseline (`____`) | `____` | `____` |
| Mi modelo (`____`) | `____` | `____` |

**Por qué la exactitud engaña aquí:** `____`

### A quién le falla

| Clase | Precisión | Recall | F1 | Soporte |
|---|---|---|---|---|
| `____` | | | | |
| `____` | | | | |

**De cada 100 detecciones difíciles, el modelo encuentra:** `____`
**Falsas alarmas producidas:** `____`

### Fuga de información

**Variable que excluí y por qué:** `____`
**Cuánto inflaba la métrica:** `____`
**La pregunta que uso para detectar este error:** `____`

### Qué aprendió el modelo

**Las dos variables más importantes:** `____`
**Mecanismo de dominio que lo explica:** `____`

### La decisión

**¿Pondrías este modelo en producción?** `____`

*(Responder "no" está permitido y a veces es lo correcto. Lo que se evalúa es el argumento, y
que esté en términos del costo del error en este dominio: qué pasa si el vehículo confía en una
detección mala, y qué pasa si es prudente de más.)*

**Qué haría falta para cambiar esa respuesta:** `____`